##Импорты

In [1]:
import numpy as np
import pandas as pd

!pip install -q gdown
import gdown

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

SEED = 42
np.random.seed(SEED)

##Загрузка и предобработка

In [2]:
file_id = "1KESjKG6IIH8PgsykdFxUms7Kkj9Zk_LY"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "signals.txt", quiet=False)

data_full = pd.read_csv('signals.txt', sep=' ', header=None, skipinitialspace=True)
signals = data_full.drop([0, 1, 2, 3, 504], axis=1)
signals.columns = [f'time_{i}' for i in range(500)]

baseline = signals.iloc[:, :60].mean(axis=1).values.reshape(-1, 1)
signals_clean = baseline - signals.values
signals_clean = np.maximum(signals_clean, 0)
signal_max = signals_clean.max(axis=1, keepdims=True)
signals_norm = signals_clean / (signal_max + 1e-8)

print(f"Данные загружены: {signals_norm.shape[0]} сигналов")

Downloading...
From: https://drive.google.com/uc?id=1KESjKG6IIH8PgsykdFxUms7Kkj9Zk_LY
To: /content/signals.txt
100%|██████████| 71.3M/71.3M [00:00<00:00, 130MB/s]


Данные загружены: 23479 сигналов


##Извлечение признаков

In [3]:
PEAK = 150
eps = 1e-8

total = signals_norm[:, PEAK-5:PEAK+100].sum(axis=1)
slow = signals_norm[:, PEAK+15:PEAK+100].sum(axis=1)
fast = signals_norm[:, PEAK+5:PEAK+25].sum(axis=1)

r_long = slow / (total + eps)
r_short = fast / (total + eps)
ratio = r_long / (r_short + eps)
asymmetry = (PEAK - np.argmax(signals_norm, axis=1)) / 50
amplitude = signals_clean.max(axis=1)
log_amp = np.log10(amplitude + 1)

X_features = np.column_stack([r_long, r_short, ratio, asymmetry, log_amp])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)
pca = PCA(n_components=3, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

print(f"Признаков: {X_features.shape[1]}, после PCA: {X_pca.shape[1]} компонент")

Признаков: 5, после PCA: 3 компонент


##Эксперимент 1: KMeans

In [6]:
print("Эксперимент 1: KMeans")
print("n_clusters | Silhouette")
print("-" * 30)

for n_clusters in [2, 3, 4]:
    kmeans = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=20)
    labels = kmeans.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels)
    print(f"{n_clusters:10d} | {sil:.4f}")

Эксперимент 1: KMeans
n_clusters | Silhouette
------------------------------
         2 | 0.3762
         3 | 0.4633
         4 | 0.4644


KMeans показал максимальный Silhouette при k=4 (0.4644), но результат при k=3 (0.4633) лишь незначительно ниже. KMeans предполагает сферические кластеры равного размера, что может не соответствовать реальной структуре данных. Однако значение 0.46 говорит о приемлемом качестве разделения.

##Эксперимент 2: Agglomerative Clustering



In [5]:
print("Эксперимент 2: Agglomerative Clustering")
print("n_clusters | linkage   | Silhouette")
print("-" * 40)

for n_clusters in [2, 3, 4]:
    for linkage in ['ward', 'complete', 'average']:
        agglo = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage)
        labels = agglo.fit_predict(X_pca)
        sil = silhouette_score(X_pca, labels)
        print(f"{n_clusters:10d} | {linkage:8} | {sil:.4f}")

Эксперимент 2: Agglomerative Clustering
n_clusters | linkage   | Silhouette
----------------------------------------
         2 | ward     | 0.3808
         2 | complete | 0.9804
         2 | average  | 0.9804
         3 | ward     | 0.4580
         3 | complete | 0.3698
         3 | average  | 0.3816
         4 | ward     | 0.4592
         4 | complete | 0.3473
         4 | average  | 0.3695


Вывод:

При k=2 с linkage='complete' или 'average' получен аномально высокий Silhouette (0.98). Это указывает на то, что алгоритм выделил один очень плотный кластер и один выброс — артефакт, а не реальное разделение.

При k=3 лучший результат показал linkage='ward' (0.4580), что сопоставимо с KMeans (0.4633).

Значения Silhouette при k=3 ниже, чем при k=2, что ожидаемо: разделение на 3 кластера сложнее, чем на 2.

Agglomerative с linkage='ward' дал стабильные результаты без артефактов, в отличие от 'complete' и 'average'.

KMeans и Agglomerative с linkage='ward' показали близкие результаты (0.4633 и 0.4580). Оба метода могут использоваться для кластеризации, но в финальном решении был выбран GMM, который после оптимизации признаков и предобработки дал результат более высокий результат.

##Эксперимент 3: DBSCAN

In [7]:
print("Эксперимент 3: DBSCAN")
print("eps | min_samples | clusters_found | Silhouette")
print("-" * 55)

for eps in [0.3, 0.5, 0.8, 1.0, 1.5]:
    for min_samples in [5, 10, 15]:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(X_pca)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        if n_clusters == 3:
            labels_fixed = np.where(labels == -1, 2, labels)
            sil = silhouette_score(X_pca, labels_fixed)
            print(f"{eps:3.1f} | {min_samples:10d} | {n_clusters:13d} | {sil:.4f}")
        else:
            print(f"{eps:3.1f} | {min_samples:10d} | {n_clusters:13d} | не 3 кластера")

Эксперимент 3: DBSCAN
eps | min_samples | clusters_found | Silhouette
-------------------------------------------------------
0.3 |          5 |            15 | не 3 кластера
0.3 |         10 |            10 | не 3 кластера
0.3 |         15 |            13 | не 3 кластера
0.5 |          5 |             2 | не 3 кластера
0.5 |         10 |             1 | не 3 кластера
0.5 |         15 |             1 | не 3 кластера
0.8 |          5 |             1 | не 3 кластера
0.8 |         10 |             1 | не 3 кластера
0.8 |         15 |             1 | не 3 кластера
1.0 |          5 |             1 | не 3 кластера
1.0 |         10 |             1 | не 3 кластера
1.0 |         15 |             1 | не 3 кластера
1.5 |          5 |             1 | не 3 кластера
1.5 |         10 |             1 | не 3 кластера
1.5 |         15 |             1 | не 3 кластера


DBSCAN не подходит для решения данной задачи по нескольким причинам:

Нестабильность числа кластеров. При eps=0.3 алгоритм находит от 10 до 15 кластеров, что не соответствует условию задачи (нужно ровно 3).

Схлопывание в один кластер. При eps≥0.5 большинство параметров приводят к выделению только одного кластера, остальные точки помечаются как шум (-1).

Отсутствие контроля числа кластеров. DBSCAN не позволяет задать фиксированное количество кластеров, что критично для данного соревнования.

Чувствительность к параметрам. Ни одна комбинация eps и min_samples не дала ровно 3 кластера, что делает метод непригодным для автоматического использования.

Вердикт: DBSCAN исключён из дальнейшего рассмотрения.

##Эксперимент 4: Gaussian Mixture Model

In [8]:
print("Эксперимент 4: Gaussian Mixture Model")
print("cov_type | reg_covar | Silhouette")
print("-" * 40)

for cov_type in ['full', 'tied', 'diag']:
    for reg_covar in [1e-4, 1e-3, 1e-5]:
        gmm = GaussianMixture(n_components=3, covariance_type=cov_type,
                              random_state=SEED, reg_covar=reg_covar, n_init=20)
        labels = gmm.fit_predict(X_pca)
        sil = silhouette_score(X_pca, labels)
        print(f"{cov_type:8} | {reg_covar:.0e} | {sil:.4f}")

Эксперимент 4: Gaussian Mixture Model
cov_type | reg_covar | Silhouette
----------------------------------------
full     | 1e-04 | 0.4008
full     | 1e-03 | 0.4009
full     | 1e-05 | 0.4008
tied     | 1e-04 | 0.3558
tied     | 1e-03 | 0.3596
tied     | 1e-05 | 0.3556
diag     | 1e-04 | 0.3816
diag     | 1e-03 | 0.3819
diag     | 1e-05 | 0.3814


Лучший результат показала модель с cov_type='full' и reg_covar=1e-03 (Silhouette 0.4009). Значения при других reg_covar практически идентичны, что говорит о стабильности модели.

cov_type='full' (разные формы и ориентации кластеров) оказалась наиболее подходящей. Это логично, так как сигналы разных типов частиц имеют различную форму затухания.

cov_type='tied' (одинаковая форма кластеров) показала худший результат (0.3558-0.3596). Жёсткое ограничение на форму кластеров снижает качество разделения.

cov_type='diag' (независимые признаки) дала промежуточный результат (0.3814-0.3819), лучше чем tied, но хуже чем full.

Стабильность. При изменении reg_covar в широком диапазоне (от 1e-05 до 1e-03) результаты практически не меняются, что подтверждает устойчивость модели.

##Общий вывод

В ходе работы были протестированы четыре метода кластеризации: KMeans, Agglomerative Clustering, DBSCAN и Gaussian Mixture Model (GMM).

**KMeans** показал лучший Silhouette Score при k=3 (0.4633). **Agglomerative Clustering** с linkage='ward' дал близкий результат (0.4580), однако при linkage='complete' и 'average' были получены аномально высокие значения (0.98), связанные с артефактным выделением кластеров. **DBSCAN** оказался непригоден для решения задачи, так как ни при одной комбинации параметров не удалось получить ровно 3 кластера.

**Gaussian Mixture Model** с cov_type='full' показала Silhouette 0.4009 — ниже, чем у KMeans и Agglomerative. Тем не менее, именно GMM была выбрана для финального решения. Это объясняется тем, что GMM позволяет моделировать кластеры разной формы и ориентации, что физически обосновано для сигналов сцинтилляционного детектора. После оптимизации предобработки (базовая линия по 60 тактам, отсечение отрицательных значений) и добавления признака asymmetry финальный результат на Kaggle составил **0.84487**.

Таким образом, наилучшее качество кластеризации достигнуто с помощью GMM при правильном подборе признаков и параметров предобработки.